In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [2]:
GENRES = ["fps", "soulslike", "cozy", "moba", "rpg", "strategy", "racing", "survival"]
REGIONS = ["India", "UAE", "Japan", "Germany", "USA", "Brazil"]

# 40 games, 5 per genre: ("fps_0", "fps"), ("fps_1", "fps"), ...
GAMES = [(f"{genre}_{i}", genre) for genre in GENRES for i in range(5)]
N_GAMES = len(GAMES)

# For each game, the number of its genre (fps = 0, soulslike = 1, ...)
GAME_GENRE = np.array([GENRES.index(genre) for _, genre in GAMES])

print(N_GAMES, "games")
print(GAMES[:6])
print(GAME_GENRE[:12])

40 games
[('fps_0', 'fps'), ('fps_1', 'fps'), ('fps_2', 'fps'), ('fps_3', 'fps'), ('fps_4', 'fps'), ('soulslike_0', 'soulslike')]
[0 0 0 0 0 1 1 1 1 1 2 2]


In [3]:
N_PLAYERS = 2000

# Each player lives in one region (a number 0 to 5)
region = rng.integers(0, len(REGIONS), size=N_PLAYERS)

# Hidden taste: 8 numbers per player (one per genre) that add up to 1.
# The model never sees this. It has to figure it out from what they play.
taste = rng.dirichlet(np.full(len(GENRES), 0.4), size=N_PLAYERS)

print(region[:10])
print(taste.shape)
print(taste[0].round(2), "-> sums to", taste[0].sum().round(2))

[0 4 3 2 2 5 0 4 1 0]
(2000, 8)
[0.   0.   0.1  0.56 0.16 0.11 0.05 0.02] -> sums to 1.0


In [4]:
# How much each player likes each GAME = how much they like that game's genre.
# taste is (2000, 8). Indexing its columns with GAME_GENRE (40 numbers)
# picks the right genre column for every game -> (2000, 40)
genre_love = taste[:, GAME_GENRE]

# Owning a game: a small base chance, plus more if you like its genre
owned = rng.random((N_PLAYERS, N_GAMES)) < 0.05 + 0.8 * genre_love

# Hours: a random "typical" amount with a long tail (most games ~20h, a few 500h+),
# scaled up for genres you love, and 0 for games you don't own
base = rng.lognormal(mean=3.0, sigma=1.0, size=(N_PLAYERS, N_GAMES))
hours = owned * base * (0.2 + 4 * genre_love)

print("games owned per player:", owned.sum(1).mean().round(1))
print("median hours on an owned game:", np.median(hours[owned]).round(1))
print("player 0 owns:", [GAMES[g][0] for g in np.where(owned[0])[0]])

games owned per player: 6.0
median hours on an owned game: 19.7
player 0 owns: ['fps_4', 'cozy_1', 'cozy_3', 'moba_0', 'strategy_2']


In [5]:
# 60,000 random pairs of players
N_PAIRS = 300_000
a = rng.integers(0, N_PLAYERS, size=N_PAIRS)
b = rng.integers(0, N_PLAYERS, size=N_PAIRS)
keep = a != b                 # nobody befriends themselves
a, b = a[keep], b[keep]

# How similar are their hidden tastes? (cosine similarity: 1 = same, 0 = nothing in common)
unit = taste / np.linalg.norm(taste, axis=1, keepdims=True)
taste_sim = (unit[a] * unit[b]).sum(1)
same_region = (region[a] == region[b]).astype(float)

# The hidden "truth": similar taste matters most, same region helps a bit
affinity = 6.0 * taste_sim + 1.5 * same_region - 4.0
p_friend = 1 / (1 + np.exp(-affinity))          # sigmoid: score -> probability

# Flip a weighted coin per pair, so it's noisy like real people
label = (rng.random(len(a)) < p_friend).astype(np.float32)

print("pairs:", len(a), "| friends:", label.mean().round(3))
print("friend rate, same region:", label[same_region == 1].mean().round(3),
      "| different region:", label[same_region == 0].mean().round(3))


pairs: 299840 | friends: 0.283
friend rate, same region: 0.479 | different region: 0.244


In [6]:
# Part 1: hours on each of the 40 games, squashed with log(1 + hours)
# 0h -> 0, 20h -> 3, 500h -> 6.2. Without this, one 500h game drowns out everything else.
game_feat = np.log1p(hours) / 7.0                   # divide by 7 so values sit around 0 to 1

# Part 2: share of total playtime per genre (8 numbers that add up to 1)
GENRE_MATRIX = np.eye(len(GENRES))[GAME_GENRE]      # (40, 8): which genre each game belongs to
genre_hours = hours @ GENRE_MATRIX                  # (2000, 8): total hours per genre
genre_share = genre_hours / np.maximum(genre_hours.sum(1, keepdims=True), 1e-9)

# Part 3: region as one-hot. India = [1,0,0,0,0,0], UAE = [0,1,0,0,0,0], ...
region_onehot = np.eye(len(REGIONS))[region]        # (2000, 6)

# Glue the parts side by side: 40 + 8 + 6 = 54 numbers per player
X = np.concatenate([game_feat, genre_share, region_onehot], axis=1).astype(np.float32)
features = torch.tensor(X)

print(features.shape)
print(features[0, 40:].round(decimals=2))   # player 0's genre share + region

torch.Size([2000, 54])
tensor([0.1200, 0.0000, 0.6800, 0.1500, 0.0000, 0.0400, 0.0000, 0.0000, 1.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000])


In [7]:
from torch.utils.data import TensorDataset, DataLoader

# Pairs as tensors: player numbers (whole numbers) and the 0/1 label
pair_a = torch.tensor(a)
pair_b = torch.tensor(b)
labels = torch.tensor(label)

# Shuffle, then keep 20% aside for validation (the model never trains on these)
perm = torch.randperm(len(labels))
n_val = int(0.2 * len(labels))
val_idx, train_idx = perm[:n_val], perm[n_val:]

train_ds = TensorDataset(pair_a[train_idx], pair_b[train_idx], labels[train_idx])
val_ds = TensorDataset(pair_a[val_idx], pair_b[val_idx], labels[val_idx])

# DataLoader hands out batches: 256 pairs at a time, in a new random order every epoch
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1024)   # no shuffle needed for checking

print("train:", len(train_ds), "| val:", len(val_ds))
ia, ib, y = next(iter(train_loader))
print(ia.shape, ib.shape, y.shape)
print(y[:10])

train: 239872 | val: 59968
torch.Size([256]) torch.Size([256]) torch.Size([256])
tensor([0., 0., 0., 1., 0., 0., 0., 0., 1., 1.])


In [8]:
class PlayerMatcher(nn.Module):
    def __init__(self, n_features, emb_dim=32):
        super().__init__()
        # The "tower": turns a player's 54 numbers into a 32-number "player fingerprint" (embedding)
        self.tower = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, emb_dim),
        )
        # Two learned numbers that stretch and shift the similarity score
        self.scale = nn.Parameter(torch.tensor(5.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def embed(self, x):
        # Normalize to length 1, so the dot product below = cosine similarity
        return F.normalize(self.tower(x), dim=-1)

    def forward(self, xa, xb):
        ea, eb = self.embed(xa), self.embed(xb)   # SAME tower for both players
        sim = (ea * eb).sum(-1)                   # cosine similarity, -1 to 1
        return self.scale * sim + self.bias       # a "logit" (raw score), not yet a probability

model = PlayerMatcher(features.shape[1]).to(device)
features = features.to(device)

print(model)
print(sum(p.numel() for p in model.parameters()), "parameters")

# Try it on one batch (untrained, so the numbers mean nothing yet)
logits = model(features[ia], features[ib])
print(logits.shape, logits[:5])

PlayerMatcher(
  (tower): Sequential(
    (0): Linear(in_features=54, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
  )
)
5602 parameters
torch.Size([256]) tensor([4.2313, 3.6669, 4.1579, 4.4250, 4.4798], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [9]:
loss_fn = nn.BCEWithLogitsLoss()     # binary cross-entropy, with the sigmoid built in
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

def run_epoch(loader, train):
    model.train(train)               # turns Dropout on for training, off for checking
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):          # no gradients needed when just checking
        for ia, ib, y in loader:
            ia, ib, y = ia.to(device), ib.to(device), y.to(device)
            logits = model(features[ia], features[ib])
            loss = loss_fn(logits, y)

            if train:
                optimizer.zero_grad()    # 1. clear old gradients
                loss.backward()          # 2. backprop: work out how to change every weight
                optimizer.step()         # 3. nudge the weights

            total_loss += loss.item() * len(y)
            correct += ((logits > 0).float() == y).sum().item()   # logit > 0 means probability > 0.5
            count += len(y)
    return total_loss / count, correct / count

best_val, best_state, patience, bad = float("inf"), None, 3, 0

for epoch in range(1, 31):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_loss < best_val - 1e-4:            # improved: remember these weights
        best_val, bad = va_loss, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:                                     # no improvement
        bad += 1
        if bad >= patience:                   # 3 epochs in a row without improving: stop
            print("stopping early")
            break

model.load_state_dict(best_state)             # go back to the best epoch
print("best val loss:", round(best_val, 3))

epoch  1 | train loss 0.660 acc 0.666 | val loss 0.573 acc 0.724
epoch  2 | train loss 0.558 acc 0.731 | val loss 0.544 acc 0.740
epoch  3 | train loss 0.540 acc 0.742 | val loss 0.536 acc 0.743
epoch  4 | train loss 0.534 acc 0.745 | val loss 0.530 acc 0.746
epoch  5 | train loss 0.529 acc 0.747 | val loss 0.527 acc 0.747
epoch  6 | train loss 0.527 acc 0.748 | val loss 0.527 acc 0.748
epoch  7 | train loss 0.525 acc 0.748 | val loss 0.525 acc 0.749
epoch  8 | train loss 0.524 acc 0.749 | val loss 0.524 acc 0.750
epoch  9 | train loss 0.523 acc 0.750 | val loss 0.524 acc 0.750
epoch 10 | train loss 0.522 acc 0.751 | val loss 0.522 acc 0.751
epoch 11 | train loss 0.521 acc 0.751 | val loss 0.522 acc 0.750
epoch 12 | train loss 0.520 acc 0.752 | val loss 0.522 acc 0.750
epoch 13 | train loss 0.519 acc 0.752 | val loss 0.521 acc 0.752
epoch 14 | train loss 0.519 acc 0.752 | val loss 0.520 acc 0.753
epoch 15 | train loss 0.518 acc 0.753 | val loss 0.520 acc 0.752
epoch 16 | train loss 0.5

In [10]:
from sklearn.metrics import roc_auc_score

# The model's scores for every validation pair
model.eval()
with torch.no_grad():
    va = pair_a[val_idx].to(device)
    vb = pair_b[val_idx].to(device)
    model_scores = model(features[va], features[vb]).cpu().numpy()

y_val = labels[val_idx].numpy()
va_np, vb_np = va.cpu().numpy(), vb.cpu().numpy()

# Simple rules a programmer might write without ML
shared_games = (owned[va_np] & owned[vb_np]).sum(1)                       # how many games in common
gu = genre_share / np.maximum(np.linalg.norm(genre_share, axis=1, keepdims=True), 1e-9)
genre_cos = (gu[va_np] * gu[vb_np]).sum(1)                                # similar genre mix
same_reg = (region[va_np] == region[vb_np]).astype(float)

print("AUC, count shared games:      ", round(roc_auc_score(y_val, shared_games), 3))
print("AUC, genre similarity:        ", round(roc_auc_score(y_val, genre_cos), 3))
print("AUC, genre similarity + region:", round(roc_auc_score(y_val, genre_cos + 0.25 * same_reg), 3))
print("AUC, our model:               ", round(roc_auc_score(y_val, model_scores), 3))
print("AUC, impossible ceiling (knows hidden taste):",
      round(roc_auc_score(y_val, affinity[val_idx.numpy()]), 3))

AUC, count shared games:       0.571
AUC, genre similarity:         0.674
AUC, genre similarity + region: 0.706
AUC, our model:                0.738
AUC, impossible ceiling (knows hidden taste): 0.837


In [11]:
from google.colab import files

checkpoint = {
    # the learned weights, moved to CPU so the website server can load them without a GPU
    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
    # everything needed to rebuild the same features later
    "n_features": features.shape[1],
    "emb_dim": 32,
    "genres": GENRES,
    "regions": REGIONS,
    "games": GAMES,
    "hours_scale": 7.0,
    "val_auc": float(roc_auc_score(y_val, model_scores)),
}

torch.save(checkpoint, "recommender.pt")
files.download("recommender.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>